# 01 — Hyper-Linear Algebra Decomposition

**GenerationalLineage companion notebook.** Toolset: `engine/toolsets/hyper_linear.py`.

Multiplication itself, read as a regular-representation matrix: `a * b` is
`b`'s digits, one tier-0 **SCALE** operator `L_d` per digit, each composed
with a **shift** `T^r` (itself just `L_base^r` — also tier-0 SCALE, by the
base). No search. `ADD` enters exactly once, at the final column-sum.

This notebook runs the toolset's own `descend()` / `build_up()` /
`verify()` live against the exact pair this whole line of investigation
grew from this session: `a=1546854629, b=7283619945`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from engine.toolsets import hyper_linear as hl
from engine.lines import AscentNotFree

A, B = 1546854629, 7283619945
d = hl.descend(A, B)
print(f"a={d['a']}  b={d['b']}")
print(f"product = {d['product']}")
print(f"spilling rows: {d['n_spilling_rows']}/10")


a=1546854629  b=7283619945
product = 11266701227799975405
spilling rows: 4/10


## The ten rows, as operators

Each row descends from exactly ONE tier-0 SCALE op (`L_d`, multiply by the
digit) composed with a shift (`T^r`, also SCALE — by the base). Nothing
richer than SCALE is needed to build any single row; ADD only enters once,
summing them.

In [2]:
for r in d["rows"]:
    print(f"  row {r['row']}: digit={r['digit']}  {r['operator']:<10}  "
          f"n_digits={r['n_digits']}  spills={r['spills']}")


  row 0: digit=5  L_5 o T^0   n_digits=10  spills=False
  row 1: digit=4  L_4 o T^1   n_digits=10  spills=False
  row 2: digit=9  L_9 o T^2   n_digits=11  spills=True
  row 3: digit=9  L_9 o T^3   n_digits=11  spills=True
  row 4: digit=1  L_1 o T^4   n_digits=10  spills=False
  row 5: digit=6  L_6 o T^5   n_digits=10  spills=False
  row 6: digit=3  L_3 o T^6   n_digits=10  spills=False
  row 7: digit=8  L_8 o T^7   n_digits=11  spills=True
  row 8: digit=2  L_2 o T^8   n_digits=10  spills=False
  row 9: digit=7  L_7 o T^9   n_digits=11  spills=True


## The DFT / convolution-theorem reconstruction

The toolset reuses `engine.spectral.dft` (not a reimplementation) to
reconstruct the product from the two zero-padded digit sequences via the
convolution theorem, then carry-propagates. Exact match, verified live —
not assumed.

In [3]:
print("DFT reconstruction:", d["dft_reconstruction"])
print("true product:        ", d["product"])
print("exact match:", d["dft_reconstruction_exact"])


DFT reconstruction: 11266701227799975405
true product:         11266701227799975405
exact match: True


## The EMERGER direction — can you get the spill pattern from the product alone?

This is the open question this notebook exists to settle, precisely: given
**only** the 20-digit product, can you determine which rows produced an
11-digit partial product?

`build_up()` implements exactly this ascent. The honest answer, checked
live below: **no, not from the bare product** — that recovery is exactly
as hard as factoring the product itself, and the toolset refuses rather
than pretend otherwise (`AscentNotFree`). Supply just ONE of the two
factors, though, and it becomes free again — one division, then the spill
pattern reads straight off the recovered factor's own digits.

In [4]:
P = d["product"]

try:
    hl.build_up({"product": P})
    print("did NOT refuse -- unexpected")
except AscentNotFree as e:
    print(f"refused, as it should: owed = {e.owed!r}")

recovered = hl.build_up({"product": P, "a": A})
print()
print(f"with one factor (a={A}) supplied:")
print(f"  recovered b = {recovered['b']}  (true b = {B})")
print(f"  spilling rows = {recovered['n_spilling_rows']}  cost = {recovered['cost']}")


refused, as it should: owed = 'at least one factor (a or b)'

with one factor (a=1546854629) supplied:
  recovered b = 7283619945  (true b = 7283619945)
  spilling rows = 4  cost = 1


## verify()

The toolset's own self-check — product exact, DFT reconstruction exact,
spill count matches the value found earlier this session (4), the bare
-product ascent genuinely refuses, and the one-factor ascent recovers
correctly.

In [5]:
import json
print(json.dumps(hl.verify(), indent=2))


{
  "ok": true,
  "ok_product": true,
  "ok_dft_reconstruction": true,
  "ok_spill_count": true,
  "ok_refuse_bare_product": true,
  "ok_recover_from_one_factor": true
}


## Where this sits in the engine's own tier vocabulary

`engine/lineage.py`'s `TIERS` table now carries `'hyper-linear'` as a
tier-0 entry, descending from SCALE alone — not a new irreducible, a
recognition that this whole construction (every row, and the shift
between rows) never needed anything above tier 0 until the very last
step, where ADD sums the rows.

In [6]:
from engine.lineage import TIERS
tier, descends, note = TIERS['hyper-linear']
print(f"tier: {tier}")
print(f"descends: {descends}")
print(f"note: {note}")


tier: 0
descends: SCALE (n-fold, one L_d per digit)
note: a*b IS n tier-0 SCALE ops (one L_d per digit of b), each composed with a shift T^r that is ALSO just SCALE (by the base) — ADD enters exactly once, summing the rows. See engine/toolsets/hyper_linear.py
